# **Data Collection**

## Objectives

* Collect the cherry-leaf image dataset from Kaggle.

* Verify the dataset structure, image counts, and dimensions.

* Confirm that the dataset contains the two required classification categories.

## Inputs

* Kaggle dataset: codeinstitute/cherry-leaves

* Dataset location: data/raw/cherry-leaves/cherry-leaves/

## Outputs

- Downloaded dataset stored under `data/raw/`.
- Verified image counts:
  - Healthy: 2,104
  - Powdery mildew: 2,104
  - Total: 4,208
- Confirmed original image dimensions of 256 × 256 pixels.

## Additional Comments

* Balanced dataset 
* Original images excluded from Git 
* API credentials are stored outside the repo


---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
from pathlib import Path
import os

PROJECT_ROOT = Path("/workspaces/FInal-Project-01")

os.chdir(PROJECT_ROOT)

print("Current working directory:", Path.cwd())

Current working directory: /workspaces/FInal-Project-01


In [2]:
current_dir = os.getcwd()
current_dir

'/workspaces/FInal-Project-01'

## Downloading the Dataset

The cherry-leaf dataset is obtained from Kaggle using the Kaggle API.

The original dataset is stored locally under `data/raw/`.
Dataset files and Kaggle API credentials are excluded from Git
to avoid committing large files or confidential information.

In [3]:
from pathlib import Path
import subprocess

dataset_directory = PROJECT_ROOT / "data" / "raw"
dataset_directory.mkdir(parents=True, exist_ok=True)

dataset_path = dataset_directory / "cherry-leaves" / "cherry-leaves"

if dataset_path.is_dir():
    print("Dataset already available:", dataset_path)
else:
    print("Dataset not found. Downloading from Kaggle...")

    subprocess.run(
        [
            "kaggle",
            "datasets",
            "download",
            "-d",
            "codeinstitute/cherry-leaves",
            "-p",
            str(dataset_directory),
            "--unzip",
        ],
        check=True,
    )

    if not dataset_path.is_dir():
        raise FileNotFoundError(
            f"Dataset download completed, but expected folder was not found: {dataset_path}"
        )

    print("Dataset downloaded successfully:", dataset_path)

Dataset already available: /workspaces/FInal-Project-01/data/raw/cherry-leaves/cherry-leaves


In [4]:
from collections import Counter
from PIL import Image

# Includes uppercase .JPG because suffixes are converted to lowercase
image_extensions = {".jpg", ".jpeg", ".png"}

image_counts = {}
image_dimensions = {}

for category in ["healthy", "powdery_mildew"]:
    folder = dataset_path / category

    if not folder.is_dir():
        raise FileNotFoundError(f"Category folder not found: {folder}")

    image_files = [
        file
        for file in folder.iterdir()
        if file.is_file()
        and file.suffix.lower() in image_extensions
    ]

    image_counts[category] = len(image_files)

    dimensions = Counter()

    for file in image_files:
        with Image.open(file) as image:
            dimensions[image.size] += 1

    image_dimensions[category] = dimensions

    print(f"\nCategory: {category}")
    print(f"Images checked: {image_counts[category]:,}")
    print(f"Image dimensions: {dict(dimensions)}")

print(f"\nTotal images: {sum(image_counts.values()):,}")


Category: healthy
Images checked: 2,104
Image dimensions: {(256, 256): 2104}

Category: powdery_mildew
Images checked: 2,104
Image dimensions: {(256, 256): 2104}

Total images: 4,208


In [5]:
from collections import Counter
from PIL import Image

for category in ["healthy", "powdery_mildew"]:
    folder = dataset_path / category

    image_files = [
        file for file in folder.iterdir()
        if file.is_file()
        and file.suffix.lower() in [".jpg", ".jpeg", ".png", "JPG"]
    ]

    dimensions = Counter()

    for file in image_files:
        with Image.open(file) as image:
            dimensions[image.size] += 1

    print(f"\nCategory: {category}")
    print(f"Image files: {len(image_files)}")
    print(f"Image dimensions: {dict(dimensions)}")


Category: healthy
Image files: 2104
Image dimensions: {(256, 256): 2104}

Category: powdery_mildew
Image files: 2104
Image dimensions: {(256, 256): 2104}


In [6]:
import pandas as pd

total_images = sum(image_counts.values())

summary_rows = []

for category in ["healthy", "powdery_mildew"]:
    dimensions = image_dimensions[category]

    summary_rows.append({
        "Category": category.replace("_", " ").title(),
        "Image Count": image_counts[category],
        "Percentage": round(
            image_counts[category] / total_images * 100, 2
        ),
        "Image Dimensions": ", ".join(
            f"{width} x {height}"
            for width, height in dimensions
        )
    })

dataset_summary = pd.DataFrame(summary_rows)

display(dataset_summary)

,Category,Image Count,Percentage,Image Dimensions
0,Healthy,2104,50.0,256 x 256
1,Powdery Mildew,2104,50.0,256 x 256


---

## Conclusions and Next Steps

### Conclusions

- The cherry-leaf dataset was obtained from Kaggle and stored under `data/raw/`.
- The dataset contains 4,208 images across two categories:
  - Healthy: 2,104 images.
  - Powdery mildew: 2,104 images.
- Both categories represent 50% of the dataset, confirming that the classes are balanced.
- All images were verified as 256 × 256 pixels.
- Image verification supports both lowercase and uppercase file extensions, including `.JPG`.
- The original images are excluded from Git version control.

### Next Steps

- Explore the visual differences between healthy and powdery mildew leaves in Notebook 02.
- Generate image montages, average images and variability images.
- Prepare the dataset for CNN training in Notebook 03.